# Resume Training (Fine-Tuning)

This notebook resumes training from `unet_lowlight_best.pth` and fine-tunes the model for an additional 30,000 steps to push PSNR higher and smooth out the loss curve.

### Key Changes:
- **Pre-loaded Weights:** Starts from the best checkpoint.
- **Lower Learning Rate:** Starts at `5e-5` instead of `2e-4` to avoid destroying learned features.
- **Mixed Loss Function:** Uses L1 + SSIM + VGG Perceptual loss for better visual quality.

In [ ]:
import os, zipfile, glob
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvmodels
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
from huggingface_hub import hf_hub_download
from collections import defaultdict

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
print('Downloading / Extracting LoL Dataset...')
try:
    zp = hf_hub_download('geekyrakshit/LoL-Dataset', 'lol_dataset.zip', repo_type='dataset')
    if not os.path.exists('lol'):
        zipfile.ZipFile(zp).extractall('lol')
except Exception as e:
    print('Dataset exists or error:', e)

pairs_dirs = defaultdict(dict)
for d in glob.glob('lol/**/', recursive=True):
    name = os.path.basename(d.rstrip('/')).lower()
    if name in ('low', 'high'):
        pairs_dirs[os.path.dirname(d.rstrip('/'))][name] = d

rows = []
for parent, dd in pairs_dirs.items():
    if 'low' in dd and 'high' in dd:
        lows = sorted(glob.glob(dd['low'] + '/*.*'))
        highs = sorted(glob.glob(dd['high'] + '/*.*'))
        for l, h in zip(lows, highs):
            rows.append({'low': l, 'high': h, 'split': 'test' if 'eval' in parent.lower() or 'test' in parent.lower() else 'train'})
df = pd.DataFrame(rows)
train = df[df.split == 'train']
test = df[df.split == 'test']

def load_pair(r):
    lo = cv2.imdecode(np.fromfile(r.low, dtype=np.uint8), cv2.IMREAD_COLOR)
    gt = cv2.imdecode(np.fromfile(r.high, dtype=np.uint8), cv2.IMREAD_COLOR)
    return lo, gt

gamma = lambda img, g: (255 * (img / 255.0) ** g).astype(np.uint8)
def prep(df_):
    X, Y = [], []
    for r in df_.itertuples():
        lo, gt = load_pair(r)
        X.append(gamma(lo, 0.4)); Y.append(gt)
    return np.stack(X), np.stack(Y)

print('Loading images into RAM...')
Xtr, Ytr = prep(train)
Xte, Yte = prep(test)
print(f'Train: {Xtr.shape}  Test: {Xte.shape}')

In [ ]:
def conv_block(inc, outc):
    return nn.Sequential(
        nn.Conv2d(inc, outc, 3, padding=1), nn.InstanceNorm2d(outc), nn.LeakyReLU(0.2, True),
        nn.Conv2d(outc, outc, 3, padding=1), nn.InstanceNorm2d(outc), nn.LeakyReLU(0.2, True),
    )

class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        c1, c2, c3, c4 = base, base*2, base*4, base*8
        self.e1 = conv_block(3, c1)
        self.e2 = conv_block(c1, c2)
        self.e3 = conv_block(c2, c3)
        self.b  = conv_block(c3, c4)
        self.u3 = nn.ConvTranspose2d(c4, c3, 2, 2)
        self.d3 = conv_block(c3 * 2, c3)
        self.u2 = nn.ConvTranspose2d(c3, c2, 2, 2)
        self.d2 = conv_block(c2 * 2, c2)
        self.u1 = nn.ConvTranspose2d(c2, c1, 2, 2)
        self.d1 = conv_block(c1 * 2, c1)
        self.out = nn.Conv2d(c1, 3, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        b  = self.b(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return (x + self.out(d1)).clamp(0, 1)

net = UNet(base=32).to(dev)
print(f'Model parameters: {sum(p.numel() for p in net.parameters()):,}')

# --- LOAD PREVIOUSLY TRAINED MODEL ---
try:
    ckpt = torch.load('unet_lowlight_best.pth', map_location=dev)
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    net.load_state_dict(state)
    print('Successfully loaded weights from unet_lowlight_best.pth!')
except Exception as e:
    print('Error loading model. Make sure unet_lowlight_best.pth is in the directory.', e)

In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, win=11, c1=0.01**2, c2=0.03**2):
        super().__init__()
        self.c1, self.c2, self.win = c1, c2, win
        self.register_buffer('kernel', torch.ones(1, 1, win, win) / win**2)

    def forward(self, x, y):
        C = x.shape[1]
        k = self.kernel.expand(C, -1, -1, -1)
        p = self.win // 2
        mu_x  = F.conv2d(x, k, padding=p, groups=C)
        mu_y  = F.conv2d(y, k, padding=p, groups=C)
        sig_x  = F.conv2d(x * x, k, padding=p, groups=C) - mu_x ** 2
        sig_y  = F.conv2d(y * y, k, padding=p, groups=C) - mu_y ** 2
        sig_xy = F.conv2d(x * y, k, padding=p, groups=C) - mu_x * mu_y
        ssim_map = ((2 * mu_x * mu_y + self.c1) * (2 * sig_xy + self.c2)) / \
                   ((mu_x**2 + mu_y**2 + self.c1) * (sig_x + sig_y + self.c2))
        return 1 - ssim_map.mean()

class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = tvmodels.vgg16(weights='IMAGENET1K_V1').features[:16].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.register_buffer('mean', torch.tensor([.485, .456, .406]).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor([.229, .224, .225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        x = (x - self.mean) / self.std
        y = (y - self.mean) / self.std
        return F.l1_loss(self.vgg(x), self.vgg(y))

ssim_loss_fn = SSIMLoss().to(dev)
vgg_loss_fn = VGGLoss().to(dev)
print('VGG and SSIM losses initialized.')

In [ ]:
# --- FINE-TUNING HYPERPARAMETERS ---
STEPS = 30_000
BS = 8
CS = 192

# Use a lower learning rate (5e-5 instead of 2e-4)
opt = torch.optim.AdamW(net.parameters(), lr=5e-5, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS, eta_min=1e-6)

def to_t(a):
    return torch.from_numpy(a).permute(0, 3, 1, 2).float().div(255).to(dev)

# Set a high initial threshold so it only saves if it beats the previous best
best_val_psnr = 21.5
H, W = Xtr.shape[1], Xtr.shape[2]

print(f'Starting fine-tuning for {STEPS} steps...')
log = []

for step in range(1, STEPS + 1):
    net.train()
    idx = np.random.randint(0, len(Xtr), BS)
    cx, cy = [], []
    
    for i in idx:
        y0 = np.random.randint(0, H - CS + 1)
        x0 = np.random.randint(0, W - CS + 1)
        cx.append(Xtr[i, y0:y0 + CS, x0:x0 + CS])
        cy.append(Ytr[i, y0:y0 + CS, x0:x0 + CS])
        
    xb = to_t(np.stack(cx))
    yb = to_t(np.stack(cy))

    # Augmentations
    if np.random.rand() < 0.5: xb, yb = xb.flip(3), yb.flip(3)
    if np.random.rand() < 0.5: xb, yb = xb.flip(2), yb.flip(2)
    if np.random.rand() < 0.3: xb, yb = xb.transpose(2, 3), yb.transpose(2, 3)

    pred = net(xb)
    
    # --- MIXED LOSS FUNCTION ---
    loss = F.l1_loss(pred, yb) + 0.3 * ssim_loss_fn(pred, yb) + 0.1 * vgg_loss_fn(pred, yb)
    
    opt.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    opt.step()
    sched.step()

    if step % 500 == 0:
        net.eval()
        with torch.no_grad():
            vpsnrs, vssims = [], []
            for j in range(len(Xte)):
                xv = to_t(Xte[j:j + 1])
                h, w = xv.shape[2], xv.shape[3]
                ph, pw = (-h) % 8, (-w) % 8
                if ph or pw: xv = F.pad(xv, (0, pw, 0, ph), mode='reflect')
                
                pv = net(xv)[:, :, :h, :w]
                pv_np = (pv[0].permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
                gt_np = Yte[j]
                
                vpsnrs.append(psnr(gt_np, pv_np, data_range=255))
                vssims.append(ssim(gt_np, pv_np, channel_axis=2, data_range=255))
                
            mpsnr, mssim = np.mean(vpsnrs), np.mean(vssims)
            
        lr = opt.param_groups[0]['lr']
        tag = ''
        if mpsnr > best_val_psnr:
            best_val_psnr = mpsnr
            torch.save(net.state_dict(), 'unet_lowlight_best.pth')
            tag = ' ★ SAVED'
            
        print(f'[{step:5d}/{STEPS}] loss={loss.item():.4f} PSNR={mpsnr:.2f} SSIM={mssim:.3f} lr={lr:.1e}{tag}')
